In [1]:
import pandas as pd
import time
from datetime import datetime, timedelta
import calendar
from openpyxl import load_workbook

In [ ]:
#pegando a data de hoje para utilizar no nome do arquivo para facilitar
data = pd.Timestamp.today()
data = str(data)

data = data.split()[0]
data = pd.Timestamp(data)
data = pd.to_datetime(data)
data = data.strftime("%m%d%Y")



print(data)

08252025


In [3]:
#Definindo cada arquivo de cada dataframe
df_p = pd.read_excel(f"PriceData_135127_UOR_{data}.xlsx")
df_p1 = pd.read_excel(f"PriceData_135127_UOR_{data} (1).xlsx")
df_p2 = pd.read_excel(f"PriceData_135127_UOR_{data} (2).xlsx")
df_site = pd.read_excel("ticket-categories-d034cd92-19c2-4152-bf7c-912d7a5517b5.xlsx")
df_depara = pd.read_excel("depara.xlsx")

In [4]:
#Juntando os 3 arquivos em um dataframe

df_universal = pd.concat([df_p, df_p1, df_p2])

Tratamento das planilhas


In [ ]:
df_universal.head()

In [5]:
#Separando o horário da data e colocando em outra coluna chamada "Temp"
df_universal[['Date', 'Temp']] = df_universal['Date'].str.split('T', expand=True)
df_site['start_date'] = df_site['start_date'].astype(str)
df_site['end_date'] = df_site['end_date'].astype(str)
df_site[['start_date', 'Temp1']] = df_site['start_date'].str.split('T', expand=True)
df_site[['end_date', 'Temp2']] = df_site['end_date'].str.split('T', expand=True)

In [6]:
#Excluindo colunas desnecessárias

df_universal.drop(['Client Number', 'Client Name', 'Sales Program', 'Sales Program Name', 'PLU', 
                   'Suggested Retail', 'Net Rate', 'Delivery Method ID', 'Pricing Type', 'Banner Color', 'Temp'], axis=1, inplace=True)


df_site.drop(['featured', 'card', 'team', 'section', 'net_price', 'net_price_teenager', 'net_price_senior', 'markup',
              'installment_additional', 'company_markup', 'company_installment_additional', 'discount', 'tax_1', 'tax_2', 'tax_3', 'tax_4',
              'tax_5', 'tax_6', 'tax_7', 'tax_8', 'tax_9', 'tax_10', 'tax_11', 'tax_12', 'Temp1', 'Temp2'], axis=1, inplace=True)

In [7]:
#fazendo a coluna "tipo"
df_universal['Tipo'] = df_universal['Product Name'].str[-5:]

In [8]:
#tentando fazer o procv..
procv = pd.merge(df_universal, df_depara, on='Product Name', how='left')

In [ ]:
df_universal['concatTitle'] = df_universal['ingressos validos']+df_universal['Date']+df_universal['Tipo']
df_universal['Valores universal'] = df_universal['Net Rate with Tax']

In [ ]:
df_site['Tipo'] = 'Adult'

#duplico todas as linhas
df_site = pd.concat([df_site, df_site], ignore_index=True)


df_site['concatTitle'] = df_site['ticket_title']+df_site['start_date']+df_site['Tipo']


In [15]:
df_universal['ingressos validos'] = procv['Unnamed: 2']

In [12]:
#Criando o arquivo da conferência do excel

data1 = pd.Timestamp.today()
data1 = str(data1)

data1 = data1.split()[0]
data1 = data1.replace('-', '.')

print(data1)

planilha = f"Conferência tabela universal teste.xlsx"
df_universal.to_excel(planilha, index=False)

2025.08.25


In [13]:
#Criando as abas no excel

with pd.ExcelWriter(planilha, engine='openpyxl', mode='a') as writer:
    df_site.to_excel(writer, sheet_name='tabela site', index=False)

with pd.ExcelWriter(planilha, engine='openpyxl', mode='a') as writer:
    df_depara.to_excel(writer, sheet_name='depara', index=False)

ld_plan = load_workbook(planilha)
ld_plan['Sheet1'].title = "tabela universal"
ld_plan.save(planilha)

In [ ]:
#procv para pegar os valores do site
procv = pd.merge(df_universal, df_site, on="concatTitle", how='left')
df_universal['Valores Site'] = procv['net_geral']